# BDB27 — outcome-link pre-flight: candidate availability, no tracking

**What this notebook is.** The stop-check that comes *before* any geometric matching: can
`player_play.csv` put an opposing route-runner on the same play as a turning DB, and how often?
It reads one small table and counts rows. It estimates nothing, fits nothing, writes nothing.

**What it is not.** It does not measure proximity, does not apply the match radius and does not
estimate an effect. The geometric step is described in §6 and is **not implemented** here — it
needs frame-level tracking this machine does not have. Nothing in this notebook is a result.

**Why a pre-flight at all.** `player_play.csv` carries rows only for the draft-prospect cohort
named in `players.csv`, so a play has as many rows as it has prospects on the field. A low
rows-per-play figure is a property of that cohort — **not** evidence that opposing players cannot
be typed by team — and no rows-per-play threshold is used anywhere here. The only conditions that
stop this notebook are real ones: **P1** (a band turn that cannot be sided) and **P2** (no
opposing route-runner anywhere in the population).

## 1. Paths — the only cell to edit

* `DATA_DIR` — the **folder** holding `player_play.csv`, and `players.csv` if you also want the
  cohort coverage line. `None` means the repo's own `data/raw`.
* `TURNS_SAMPLE` — the band-turn table to pre-flight. `None` means the repo's
  `outputs/turns/turns.parquet` (the 2023 sample turns).
* `TURNS_FULL` — an optional second band-turn table. `None` means the repo's
  `data/interim/full_run_20261111/bdb27_full_db_turns.parquet`, used when present. On a machine
  without it the FULL population is simply not measured and the notebook says so.
* `ROWS_TO_SHOW` — how many per-turn rows §5 prints.

Nothing below §1 needs editing. `game_tracking_2023_sample.csv` is **not** an input to this
notebook and must never become one: it cannot characterise availability.

In [ ]:
# ======================= CONFIG — the only cell to edit =========================
DATA_DIR    = None    # folder holding player_play.csv (+ players.csv); None = repo data/raw
TURNS_SAMPLE = None   # None = repo outputs/turns/turns.parquet
TURNS_FULL   = None   # None = repo data/interim/full_run_20261111/bdb27_full_db_turns.parquet
ROWS_TO_SHOW = 10     # per-turn rows printed in §5
# =============================================================================

## 2. Setup

Finds the repo (walks up until it sees `config.yaml` and `src/bdb27`), puts `src/` on the path and
imports `bdb27.outcome_link` — the same module `scripts/outcome_link_feasibility.py` uses, so the
candidate definition and the P1–P5 verdicts exist in exactly one place. Then opens a scratch
DuckDB connection under the project's memory limit.

The frozen design constants are printed from the module, not retyped here: the match radius and
the two gate thresholds are the note's, unchanged.

In [ ]:
import importlib
import sys
import textwrap
from pathlib import Path


def find_repo_root() -> Path:
    '''Walk upward from the notebook's working directory.'''
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "config.yaml").exists() and (parent / "src" / "bdb27").is_dir():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/bdb27). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from bdb27 import outcome_link

con = outcome_link.connect()
print(f"repo            : {REPO}")
print(f"shared module   : {Path(outcome_link.__file__).resolve()}")
print(f"match radius    : {outcome_link.PRIMARY_RADIUS_YD} yd (frozen design choice)")
print(f"gates           : >={outcome_link.GATE_MIN_ATTRIBUTED_TURNS} turns "
      f"and >={outcome_link.GATE_MIN_SHARE:.0%} of band turns")
print(f"band            : {outcome_link.BAND_LO:g}-{outcome_link.BAND_HI:g} degrees, left-closed")
print(f"turn key        : {', '.join(outcome_link.TURN_KEY)}")

## 3. Inputs

`player_play.csv` is the only data input. `players.csv` is optional and only adds the cohort
coverage line. At least one band-turn table is required — without turns there is nothing to
pre-flight, and that is the one input condition that stops the notebook.

Every path is printed with a present/absent marker before anything is read; nothing is silently
substituted.

In [ ]:
raw_dir = (Path(DATA_DIR).expanduser() if DATA_DIR else REPO / "data" / "raw").resolve()
pp_csv = raw_dir / "player_play.csv"
players_csv = raw_dir / "players.csv"

turns_sample = (Path(TURNS_SAMPLE).expanduser() if TURNS_SAMPLE
                else REPO / "outputs" / "turns" / "turns.parquet")
turns_full = (Path(TURNS_FULL).expanduser() if TURNS_FULL
              else REPO / "data" / "interim" / "full_run_20261111"
                   / "bdb27_full_db_turns.parquet")

print(f"{'input':<28}{'present':<9}path")
for label, path in (("player_play.csv (required)", pp_csv),
                    ("players.csv (optional)", players_csv),
                    ("band turns [SAMPLE]", turns_sample),
                    ("band turns [FULL] (optional)", turns_full)):
    print(f"{label:<28}{'yes' if path.exists() else 'no':<9}{path}")

if not pp_csv.exists():
    raise RuntimeError(
        f"player_play.csv is absent: {pp_csv}\n"
        "  Set DATA_DIR in §1 to the folder holding it. Nothing was read.")
if not turns_sample.exists() and not turns_full.exists():
    raise RuntimeError(
        "No band-turn table found, so there is nothing to pre-flight:\n"
        f"  SAMPLE: {turns_sample}\n  FULL  : {turns_full}\n"
        "  Set TURNS_SAMPLE / TURNS_FULL in §1. Nothing was read.")

outcome_link.load_player_play(con, pp_csv)
print()
print("player_play rows:", f"{outcome_link.availability(con)['pp_rows']:,}")
if players_csv.exists():
    con.execute(
        "CREATE OR REPLACE TABLE ol_players AS SELECT TRY_CAST(nfl_id AS BIGINT) AS nfl_id "
        f"FROM read_csv('{players_csv.as_posix()}', header=true, all_varchar=true, "
        "nullstr=['NA',''])")
    AVAIL = outcome_link.availability(con, players_table="ol_players")
else:
    AVAIL = outcome_link.availability(con)
for key, value in AVAIL.items():
    print(f"  {key:<34}{value}")

BAND = {}
for stamp, path in (("SAMPLE", turns_sample), ("FULL", turns_full)):
    if not path.exists():
        print(f"\n{stamp}: band-turn table absent — this population is not measured here.")
        continue
    table = f"band_{stamp.lower()}"
    n = outcome_link.register_band_turns(con, path, table)
    keys = outcome_link.check_turn_key(con, table)
    BAND[stamp] = table
    print(f"\n{stamp}: {n:,} band turns")
    print(f"  on the turn key {outcome_link.TURN_KEY}: {keys['rows_turn_key']:,} rows")
    print(f"  grouping on (game_id, play_id, nfl_id) alone would keep "
          f"{keys['rows_player_key']:,} and silently drop "
          f"{keys['second_turns_dropped_by_player_key']:,} second turns")

## 4. Pre-flight — P1 to P5

Run on `player_play` alone: no tracking file is opened, and `outcome_link.preflight` takes no
tracking argument by design.

| | condition | what happens |
|---|---|---|
| **P1** | a band turn whose DB has no `team_abbr`, no `player_play` row, or two conflicting labels | **stop** — the turn cannot be sided; the turns are named |
| **P2** | zero turns with an opposing route-runner | **stop** — the route is impossible in this dataset |
| **P3** | ceiling below the count gate | estimator refused; the measurement still prints |
| **P4** | ceiling below the share gate | `GATE UNREACHABLE — descriptive only`; no estimate |
| **P5** | how much of the ceiling the prospect cohort hides | always printed |

The verdict is the module's, not this cell's. This cell stops only when `stop` is true, i.e. only
on P1 or P2.

In [ ]:
PRE, CAND = {}, {}
for stamp, table in BAND.items():
    cand_table = f"cand_{stamp.lower()}"
    CAND[stamp] = outcome_link.candidate_table(con, table, cand_table, stamp)
    PRE[stamp] = outcome_link.preflight(con, cand_table, stamp)

stops = []
for stamp in BAND:
    p = PRE[stamp]
    print(f"=== {stamp}: {p['band_turns']:,} band turns")
    print(f"  P1 siding   : {p['p1_unsided_turns']:,} unsided "
          f"(no player_play row {p['p1_turns_without_player_play_row']:,}, "
          f"no team label {p['p1_turns_without_team_label']:,}, "
          f"conflicting label {p['p1_turns_with_conflicting_team_label']:,})")
    print(f"  P2 pool     : {p['p2_turns_with_candidate']:,} turns with at least one "
          f"opposing route-runner ({p['p2_ceiling_share']:.1%}) — the ceiling on unique")
    print(f"  P3 count    : {p['p3_gate_min_turns']:,} turns required — ceiling "
          f"{'reaches it' if p['p3_cleared'] else 'FALLS SHORT OF IT'} "
          f"(ceiling is {p['p2_turns_with_candidate']:,})")
    print(f"  P4 share    : {p['p4_gate_min_share']:.0%} of band turns required "
          f"({p['p4_required_turns']:,} turns) — ceiling "
          f"{'reaches it' if p['p4_cleared'] else 'FALLS SHORT OF IT'} "
          f"(ceiling share is {p['p2_ceiling_share']:.1%});")
    print("                a reachable ceiling is NOT a passed gate — neither gate is")
    print("                decided until the frozen radius is applied to real positions.")
    print("  P5 cohort ceiling:")
    for line in textwrap.wrap(p["p5_wording"], width=96):
        print(f"    {line}")
    print(f"  verdict     : {p['verdict']}")
    if p["stop"]:
        stops.append(stamp)
    print()

if stops:
    detail = "\n".join(
        f"  {s}: {PRE[s]['verdict']}; turns: "
        f"{'; '.join(PRE[s]['p1_unsided_turn_keys'][:20]) or '(none listed)'}"
        for s in stops)
    raise RuntimeError(
        "PRE-FLIGHT STOPPED on a real condition (P1 or P2):\n" + detail +
        "\n  No estimate can be produced from this population. Nothing was written.")

print("Pre-flight complete: no real stop condition fired in any population measured.")

## 5. Candidate availability by band turn

One row per band turn: the turning DB's own team label, how many opposing players the play has in
`player_play`, how many of those ran a route, how many of those carry a separation value, and the
status `none_on_play` / `single` / `multiple`. The totals below are the candidate availability of
§4 read per turn.

**Ties are counted and never resolved.** Where two candidates appear, nothing here prefers one:
which receiver the ball went to is a post-hoc outcome and may not break a tie.

In [ ]:
DETAIL = {}
for stamp in BAND:
    print(f"=== {stamp}")
    df = CAND[stamp]
    for row in outcome_link.status_totals(df, stamp):
        print(f"  {row['pp_candidate_status']:<14}{row['n_turns']:>8,}"
              f"{row['share_of_turns']:>9.1%}  of {row['total_turns']:,} turns")
    print(f"  turns with at least one candidate carrying a separation value: "
          f"{int((df['pp_opp_route_sep_rows'] >= 1).sum()):,}")

    detail = outcome_link.candidate_detail(con, f"cand_{stamp.lower()}",
                                           f"detail_{stamp.lower()}")
    DETAIL[stamp] = detail
    ties = outcome_link.ambiguity_counts(detail)
    print("  ties (counted, never broken):")
    print(f"    turns with two candidates              : {ties['turns_with_two_candidates']:,}")
    print(f"    ... both carry a separation value      : "
          f"{ties['two_candidate_turns_with_both_separations']:,}")
    print(f"    ... a target row exists (post-hoc, unused): "
          f"{ties['two_candidate_turns_with_a_target_row']:,}")
    gap_min = ties["two_candidate_separation_gap_min_yd"]
    gap_med = ties["two_candidate_separation_gap_median_yd"]
    gap_max = ties["two_candidate_separation_gap_max_yd"]
    if gap_min is None:
        print("    separation gap (yd)                    : no pair carries two values")
    else:
        print(f"    separation gap (yd) min / median / max : {gap_min} / {gap_med} / {gap_max}")
    print()
    print(df.head(ROWS_TO_SHOW).to_string(index=False))
    print()

## 6. What the geometric step must still do — **not implemented**

This notebook stops at availability. What is left needs frame-level tracking and cannot be
measured on this machine; it is described here so the gap is explicit, not so it can be faked.

1. **Break frame.** For every band turn, recover the break timestamp from `(x_centre, y_centre)`
   against the player's own frames, with the same ≤0.25 yd alignment gate the sample uses.
2. **Candidates.** Every player in the frame at that timestamp; team from `player_play`;
   opposing **route-runners** as the primary candidate set, any opponent as the fallback.
3. **Apply the frozen rule.** A turn is attributed to **the single opposing route-runner within
   3 yd** at the break frame. Two or more candidates inside the radius → ambiguous, **dropped**;
   none inside → unmatched, **dropped**; no tie is broken by radius choice, by distance order or
   by row order. The rule is frozen and this notebook does not change it.
4. **Persistence.** At the pass-forward frame, check whether the matched DB is the nearest
   defender to that receiver; if not, the separation belongs to another defender's coverage.
5. **Gates, before any estimate.** The note pre-declares two: at least **500** attributed turns,
   and at least **30%** of band turns unique. Below either, the output is descriptive only. §4
   reports whether each ceiling is reachable, at the population level, *before* any geometry.
6. **The tie rate is unmeasured, not zero.** The 2023 sample file cannot put two receivers in one
   frame often enough to say anything about ties; §5 counts the ties that exist in
   `player_play` and says nothing about proximity. Only full frame-level data can measure the
   tie rate, and until it does, no estimate may be written.

No model, no coefficient, no interval, no finding is produced anywhere in this notebook.